[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlatcl/multilogue/blob/main/notebooks/09_formats_complexity.ipynb)

# Formats complexity — comparing modes of convening

### [Neil D. Lawrence](http://inverseprobability.com), University of Cambridge

### 2026-10-16

**Abstract**: Public-dialogue studies often ask whether *how* people are convened changes
*what* they raise — workshop versus survey, chatbot versus in-person. Multilogue
does not hard-code “format”; it supplies the Address machinery with a group
column and a consumer metric $\ell$. This notebook teaches that pattern: Access
on a public format pair, then Address that prefers a cached CIMT LLM run
(resuming via StageRunner when an API key is present), with a FakeClient
formats demo as the offline fallback — soft contrasts, document EMD, and a
multiverse over analytic decisions.


In [ ]:
#@title Setup (run first)
# Infrastructure only: install or locate multilogue, prepare figure directory.
# Scientific imports live in later cells.

import sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "matplotlib", "pandas", "numpy", "scikit-learn", "pyyaml", "pot",
        "git+https://github.com/mlatcl/multilogue.git",
    ])
else:
    try:
        import multilogue  # noqa: F401
    except ImportError:
        _here = pathlib.Path.cwd().resolve()
        for _cand in (_here, _here.parent, *_here.parents):
            if (_cand / "pyproject.toml").is_file() and (_cand / "multilogue").is_dir():
                if str(_cand) not in sys.path:
                    sys.path.insert(0, str(_cand))
                break

from multilogue.notebook import setup

FIG = setup(figures_dir="outputs/notebook_figures")
print("Figures →", FIG)


# Introduction

An `ai-dialogue-formats`-style study keeps PDF parsers, prompts, and the precise
definition of $\ell$ in the consumer package. Multilogue owns extract → cluster →
weight → soft lenses → document distance → multiverse. The group column here is
`format` $\in\{\mathrm{dialogue},\mathrm{survey}\}$. The headline $\ell$ is a
document-weighted format gap on the lens with largest absolute difference —
recomputed under each analytic configuration without new LLM calls.


In [ ]:
# Optional transport dependency (soft lenses + document EMD).
import importlib.util, subprocess, sys
if importlib.util.find_spec("ot") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "pot"])
print("POT available")


## Access — public format pair

CIMT Bonn cycling *dialogue* vs *survey* stand in for workshop vs survey when
the network is available. Assess without the analysis question: coverage by
format and unit sizes. If CIMT cannot be fetched, we fall back to a short
synthetic Access table so the notebook still runs.


In [ ]:
import os
from IPython.display import display
import pandas as pd
from multilogue.experiment import load_cimt_format_pair, format_tagged_units
from multilogue.assess_base import docs_per_group, unit_size_table


In [ ]:
os.environ.setdefault("MULTILOGUE_ACCESS_YES", "1")
cimt_units, cimt_meta = load_cimt_format_pair(max_units_per=60)
if cimt_units is None:
    print("CIMT unavailable:", cimt_meta.get("error", "unknown"))
    access_units = format_tagged_units(n_docs_per_group=3, units_per_doc=2)
    access_label = "synthetic format stand-in (Access fallback)"
else:
    access_units = cimt_units
    access_label = "CIMT Cycling-Dialogue-Bonn vs Survey-Bonn"
    display(cimt_meta)

print(access_label, "—", len(access_units), "units")
display(docs_per_group(access_units, "format"))
display(unit_size_table(access_units).describe())


## Address — cache first, then FakeClient fallback

StageRunner checkpoints under `outputs/formats_cimt_llm/` mean a prior CIMT LLM
run can be reused without paying again. The next cell therefore:

1. loads `.env` if present;
2. if `OPENAI_API_KEY` is set and CIMT units exist — *resume or recache* the live
   demo (completed fingerprints are skipped);
3. else if a complete cached run is already on disk — use those tables;
4. else run the offline FakeClient formats demo so Colab / CI still work.

German CIMT text does not match the English FakeClient phrase bank, so the
synthetic fallback keeps the same `format` group mechanics without an API.


In [ ]:
import json
import os
from pathlib import Path
from IPython.display import display, Markdown
from multilogue.config import load_dotenv
from multilogue.experiment import run_formats_complexity_demo
from multilogue import viz
from multilogue.report import (
    cluster_catalogue, findings_markdown, top_phrases_by_group, write_findings,
)


In [ ]:
load_dotenv(Path.cwd() / ".env")
load_dotenv(Path.cwd().parent / ".env")
# Prefer repo-root outputs when the notebook cwd is notebooks/.
for _root in (Path.cwd(), Path.cwd().parent):
    _cand = _root / "outputs" / "formats_cimt_llm"
    if (_cand / "tables" / "phrases_clustered.csv").is_file() or (_cand / "checkpoints").is_dir():
        LLM_OUT = _cand
        break
else:
    LLM_OUT = Path("outputs/formats_cimt_llm")

CACHE_TABLES = LLM_OUT / "tables"
cache_ready = (CACHE_TABLES / "phrases_clustered.csv").is_file()
key = os.environ.get("OPENAI_API_KEY", "").strip()
summary = None
address_source = None

if key and cimt_units is not None:
    from multilogue.openai_client import OpenAIClient
    units_live = [
        u for u in cimt_units
        if (u.get("meta") or {}).get("code") != "non-arg"
    ]
    print(
        f"Live / resume CIMT Address → {LLM_OUT} "
        f"({len(units_live)} units; checkpoints reused when fingerprints match)"
    )
    summary = run_formats_complexity_demo(
        LLM_OUT,
        units=units_live,
        client=OpenAIClient(log_path=LLM_OUT / "logs" / "openai.jsonl"),
        kinds=("concern", "benefit"),
        extract_model="gpt-4.1-mini",
        embed_model="text-embedding-3-small",
        label_model="gpt-4.1-mini",
        k=6, lens_runs=3, quiet=True, max_workers=4,
    )
    tables = CACHE_TABLES
    address_source = "cimt_llm"
elif cache_ready:
    print(f"Using cached CIMT Address tables → {CACHE_TABLES.resolve()}")
    tables = CACHE_TABLES
    address_source = "cimt_cache"
    man = LLM_OUT / "run_manifest.json"
    if man.is_file():
        summary = json.loads(man.read_text())
        display(summary)
else:
    out = Path("outputs/notebook_formats_complexity")
    print(
        "No CIMT cache and no API key — FakeClient formats demo →", out
    )
    summary = run_formats_complexity_demo(
        out, n_docs_per_group=4, units_per_doc=3, k=5, lens_runs=3, quiet=True,
    )
    tables = out / "tables"
    address_source = "fakeclient"

display(summary)
print("Address source:", address_source)
print("Tables:", tables.resolve())
print("Checkpoints:", LLM_OUT / "checkpoints" if address_source != "fakeclient" else tables.parent / "checkpoints")


In [ ]:
for name in [
    "yield.csv",
    "group_compare_format_lenses.csv",
    "soft_group_compare_format.csv",
    "sigma2_rules.csv",
    "doc_distance_pairwise.csv",
    "variance_shares_format_gap.csv",
]:
    p = tables / name
    if p.is_file():
        print("—", name)
        display(pd.read_csv(p).head(12))

shares_path = tables / "variance_shares_format_gap.csv"
if shares_path.is_file():
    shares = pd.read_csv(shares_path)
    viz.variance_share_bars(shares, path=FIG / "09_format_gap_variance.png")
if isinstance(summary, dict):
    print("Soft:", summary.get("soft"))
    print("Distance:", summary.get("distance"))


## Reading the run

Findings summarise clusters, top concerns by format, and which analytic
decisions move $\ell$. This cell only reads tables — no LLM calls.


In [ ]:
ph = pd.read_csv(tables / "phrases_clustered.csv")
cmp = (
    pd.read_csv(tables / "group_compare_format_lenses.csv")
    if (tables / "group_compare_format_lenses.csv").is_file() else None
)
multi = (
    pd.read_csv(tables / "multiverse_format_gap.csv")
    if (tables / "multiverse_format_gap.csv").is_file() else None
)
shares = (
    pd.read_csv(tables / "variance_shares_format_gap.csv")
    if (tables / "variance_shares_format_gap.csv").is_file() else None
)
display(Markdown("### Cluster catalogue"))
display(cluster_catalogue(ph))
display(Markdown("### Top concerns by format"))
for g, tab in top_phrases_by_group(ph, kind="concern", n=8).items():
    display(Markdown(f"**{g}**"))
    display(tab)
if shares is not None:
    display(Markdown("### Multiverse variance shares"))
    display(shares)
title = (
    "CIMT dialogue vs survey — findings"
    if address_source in {"cimt_llm", "cimt_cache"}
    else "Dialogue vs survey — findings (FakeClient)"
)
md_text = findings_markdown(
    ph, compare=cmp, multiverse=multi, variance_shares=shares, title=title,
)
write_findings(
    ph, tables / "findings.md", compare=cmp,
    multiverse=multi, variance_shares=shares, title=title,
)
display(Markdown(md_text))


# Outlook

Multilogue owns the shared stages above. A formats-style consumer package still
keeps: PDF/docx Access, study prompts and experience coding, researcher-edited
lens YAML, and the precise definition of headline $\ell$. See the formats
cookbook and the soft / EMD concepts page in the documentation.
